In [2]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from qdrant_client import models, QdrantClient
import pandas as pd

In [ ]:
df = pd.read_csv('Olive_Oil_Sensory_Dataset_International.csv')
df = df[df['name'].notna()] # remove any NaN values as it blows up serialization
data = df.sample(800).to_dict('records') # Get only 800 records. More records will make it slower to index
len(data)

20

In [4]:
encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [5]:
# create the vector database client
qdrant = QdrantClient(":memory:")  # Create in-memory Qdrant instance

In [6]:
# Check if collection exists
if qdrant.collection_exists("top_olive_oils"):
    qdrant.delete_collection("top_olive_oils")

# Create collection
qdrant.create_collection(
    collection_name="top_olive_oils",
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(),
        distance=models.Distance.COSINE
    )
)


True

In [7]:
# vectorize!
qdrant.upload_points(
    collection_name="top_olive_oils",
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(doc["description"]).tolist(),
            payload=doc,
        ) for idx, doc in enumerate(data)  # data is the variable holding all the olive oils
    ]
)

In [8]:
user_prompt = "Suggest me an olive oil with fruity aroma and mild bitterness."

In [9]:
# Search time for awesome olive oils!

hits = qdrant.query_points(
    collection_name="top_olive_oils",
    query=encoder.encode(user_prompt).tolist(),
    limit=3
)

# Iterate over the results
for point in hits.points:
    print(point.payload, "score:", point.score)



{'id': 'INT-000458', 'name': 'Corfiot Lianolia • Trás-os-Montes', 'region': 'Trás-os-Montes', 'subregion_or_pdo_pgi': 'Beira Interior', 'cultivars': 'Koroneiki', 'harvest_year': 2023.9943916639256, 'category': 'EVOO', 'color': 'green-gold', 'fruitiness_intensity': 7.491115967760114, 'bitterness_intensity': 3.772879950218224, 'pungency_intensity': 5.619409552134313, 'aroma_notes': 'green walnut;wild fennel;apple skin', 'taste_notes': 'silky;fresh;medium pepper', 'defects': 'none', 'processing': 'cold-extracted', 'filtration': 'filtered', 'acidity_pct': 0.3219621329160992, 'awards': '—', 'description': 'Elegant Corfiot oil with herbal lift; shines on steamed vegetables and fish.'} score: 0.6069398588154886
{'id': 'INT-000696', 'name': 'Laconian Peaks • Sousse', 'region': 'Sousse', 'subregion_or_pdo_pgi': 'Sousse', 'cultivars': 'Mission', 'harvest_year': 2024.013006009808, 'category': 'EVOO', 'color': 'bright green', 'fruitiness_intensity': 6.607534715069903, 'bitterness_intensity': 4.979

In [10]:
# Store payloads
search_results = [point.payload for point in hits.points]

In [11]:
# Connect with the local large language model
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

completion = client.chat.completions.create(
    model="qwen2.5:3b",
    messages=[
        {"role":"system", "content":"You are a chatbot, an olive oil specialist. Your top priority is to help guide users into selecting amazing oil and guide them with their requests. Provide me the answer in pure text"},
        {"role":"user", "content":"Suggest me an olive oil with fruity aroma and mild bitterness."},
        {"role":"assistant", "content": str(search_results)},
    ]
)
print(completion.choices[0].message)

ChatCompletionMessage(content="  \n\nThe Corfiot Lianolia from the Trás-os-Montes region seems to fit your description of having a fruity aroma and mild bitterness. Its aroma notes include green walnut, wild fennel, apple skin, which are characteristic of fruits. The bitter components such as green walnut might contribute to the mild bitterness you're looking for.\n\nCould it be perfect? Maybe not every oil is right for everyone. But this one strikes a balance between your desired features. You can further check its taste and aroma according to your preference by considering other options in the list or visiting nearby olive groves if available. Enjoy exploring!", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
